# Deepfake Hybrid Detection — Colab Training Notebook

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

### Dataset structure expected
```
ffpp-dataset/
├── original_sequences/youtube/c23/videos/   ← real videos (.mp4)
└── manipulated_sequences/
    ├── Deepfakes/c23/videos/                ← fake videos (.mp4)
    └── Face2Face/c23/videos/               ← fake videos (.mp4)
```

### Pipeline steps
1. Mount Google Drive & clone project code
2. Install dependencies
3. Copy dataset from Drive to local `/content/` (faster I/O)
4. Update `config.yaml` paths
5. Extract frames from videos
6. Build train/val/test splits
7. Compute FFT cache
8. Train models
9. Evaluate & save outputs back to Drive
10. Save everything back to Drive

## Step 1 — Mount Drive & clone project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ---------------------------------------------------------------------------
# OPTION A: your code is on GitHub — replace with your actual repo URL
# ---------------------------------------------------------------------------
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/skripsi

# ---------------------------------------------------------------------------
# OPTION B: your code is already in Google Drive
#   Change the path below to where your project folder lives in Drive
# ---------------------------------------------------------------------------
CODE_IN_DRIVE = "/content/drive/MyDrive/skripsi"   # <-- EDIT THIS
!cp -r "{CODE_IN_DRIVE}" /content/skripsi

os.chdir("/content/skripsi/deepfake_hybrid")
!pwd && ls

## Step 2 — Install dependencies

In [ ]:
!pip install -q timm scikit-learn tqdm opencv-python-headless
# ffmpeg is needed so OpenCV can decode H.264/H.265 .mp4 files
!apt-get install -qq ffmpeg libavcodec-extra
# torch + torchvision are pre-installed on Colab GPU runtimes
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Step 3 — Copy dataset from Drive to local storage

Reading directly from Drive during training is very slow.
Copy the dataset to `/content/` first (fast local SSD).

**Expected FFPP folder structure in your Drive:**
```
ffpp-dataset/
├── original_sequences/
│   └── youtube/
│       └── c23/
│           └── videos/          ← real .mp4 files here
└── manipulated_sequences/
    ├── Deepfakes/
    │   └── c23/
    │       └── videos/          ← fake .mp4 files here
    └── Face2Face/
        └── c23/
            └── videos/          ← fake .mp4 files here
```

**How to get the dataset into your Drive:**
1. Open the shared folder link
2. Click the dropdown next to the folder name → *Add shortcut to Drive*
3. Place it anywhere in *My Drive* and note the path

In [ ]:
# Path to the top-level FFPP folder in your Drive
# This folder must contain both original_sequences/ and manipulated_sequences/
FFPP_IN_DRIVE = "/content/drive/MyDrive/ffpp-dataset"   # <-- EDIT THIS

# Verify the expected subfolders exist before copying
import os

real_path = os.path.join(FFPP_IN_DRIVE, "original_sequences", "youtube", "c23", "videos")
fake_df_path = os.path.join(FFPP_IN_DRIVE, "manipulated_sequences", "Deepfakes", "c23", "videos")
fake_f2f_path = os.path.join(FFPP_IN_DRIVE, "manipulated_sequences", "Face2Face", "c23", "videos")

for label, path in [("real (original)", real_path), ("fake (Deepfakes)", fake_df_path), ("fake (Face2Face)", fake_f2f_path)]:
    exists = os.path.isdir(path)
    count = len([f for f in os.listdir(path) if f.endswith(".mp4")]) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} [{label}] {path} — {count} .mp4 files")

In [ ]:
# Copy to local SSD — this may take a few minutes depending on dataset size
!cp -r "{FFPP_IN_DRIVE}" /content/ffpp_data
print("Done copying. Top-level contents:")
!ls /content/ffpp_data
print("\nReal videos:")
!ls /content/ffpp_data/original_sequences/youtube/c23/videos | head -5
print("\nFake videos (Deepfakes):")
!ls /content/ffpp_data/manipulated_sequences/Deepfakes/c23/videos | head -5
print("\nFake videos (Face2Face):")
!ls /content/ffpp_data/manipulated_sequences/Face2Face/c23/videos | head -5

## Step 4 — Update config.yaml paths

In [ ]:
import yaml
from pathlib import Path

CONFIG_PATH = "/content/skripsi/deepfake_hybrid/config.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Root points to the top-level ffpp folder containing both
# original_sequences/ and manipulated_sequences/
cfg['ffpp_root'] = "/content/ffpp_data"
cfg['datasets']['ffpp']['root'] = "/content/ffpp_data"

# Label inference uses path parts — these keywords match the actual folder names:
#   real:  ...original_sequences/youtube/...  → matched by "youtube" / "original"
#   fake:  ...manipulated_sequences/Deepfakes/... → matched by "deepfakes"
#   fake:  ...manipulated_sequences/Face2Face/... → matched by "face2face"
cfg['datasets']['ffpp']['real_keywords'] = ["original", "real", "pristine", "actors", "youtube"]
cfg['datasets']['ffpp']['fake_keywords'] = ["fake", "manipulated", "deepfakes", "faceswap",
                                             "neuraltextures", "deepfakedetection", "faceshifter",
                                             "face2face"]

# Write outputs to local SSD (we'll sync to Drive at the end)
cfg['output_root'] = "/content/outputs"

# Recommended: increase epochs for a real training run
cfg['epochs'] = 20          # change to 30-50 for a full run
cfg['n_seeds'] = 1          # increase to 3 for statistical validity
cfg['batch_size'] = 16      # reduce to 8 if you get OOM errors
cfg['num_workers'] = 2      # Colab works well with 2

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f)

print("Config updated:")
with open(CONFIG_PATH) as f:
    print(f.read())

## Step 5 — Extract frames from videos

`extract_frames.py` recursively scans `ffpp_data/` for `.mp4` files and infers labels from folder names:
- `original_sequences/youtube/...` → **real** (label 0)
- `manipulated_sequences/Deepfakes/...` → **fake** (label 1)
- `manipulated_sequences/Face2Face/...` → **fake** (label 1)

**If you get an empty manifest error**, run the debug cell below first to diagnose.

In [ ]:
!python /content/skripsi/deepfake_hybrid/scripts/extract_frames.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --fps 5 \
    --max-frames 100

# Sanity check: manifest must be non-empty before continuing
import pandas as pd
from pathlib import Path

manifest_path = "/content/outputs/manifests/FFPP/manifest.csv"
assert Path(manifest_path).exists(), "manifest.csv was not created — extract_frames.py may have crashed"

manifest = pd.read_csv(manifest_path)

if len(manifest) == 0:
    raise RuntimeError(
        "manifest.csv is empty — no videos were successfully processed.\n"
        "Possible causes:\n"
        "  1. The copy in Step 3 failed or is still in progress\n"
        "  2. OpenCV couldn't decode the videos (run: !python -c \"import cv2; "
        "cap=cv2.VideoCapture('/content/ffpp_data/original_sequences/youtube/c23/videos/000.mp4'); "
        "print(cap.isOpened())\")\n"
        "  3. The videos folder path inside ffpp_data is different from expected\n"
        "     (run: !find /content/ffpp_data -name '*.mp4' | head -5)"
    )

print(f"Total videos in manifest: {len(manifest)}")
print(f"  Real (label=0): {manifest['label'].eq(0).sum()}")
print(f"  Fake (label=1): {manifest['label'].eq(1).sum()}")
if manifest['label'].isna().any():
    print(f"  WARNING: {manifest['label'].isna().sum()} videos had unknown labels and were skipped")

# Warn if real/fake ratio is very unbalanced
real_n = manifest['label'].eq(0).sum()
fake_n = manifest['label'].eq(1).sum()
if real_n == 0:
    raise RuntimeError("0 real videos found — check that original_sequences/youtube/c23/videos/ has .mp4 files")
if fake_n == 0:
    raise RuntimeError("0 fake videos found — check that manipulated_sequences/Deepfakes and Face2Face folders have .mp4 files")

In [ ]:
# ── DEBUG (run this if Step 5 gives an empty manifest) ──────────────────────
# 1. Confirm .mp4 files are actually present
print("=== .mp4 files found under /content/ffpp_data ===")
!find /content/ffpp_data -name "*.mp4" | head -10
print()

# 2. Check OpenCV can open one of them
import cv2, glob
mp4s = glob.glob("/content/ffpp_data/**/*.mp4", recursive=True)
if mp4s:
    cap = cv2.VideoCapture(mp4s[0])
    print(f"OpenCV can open {mp4s[0]}: {cap.isOpened()}")
    cap.release()
else:
    print("No .mp4 files found — copy may have failed")
# ─────────────────────────────────────────────────────────────────────────────

## Step 6 — Build train / val / test splits

In [ ]:
import pandas as pd
from pathlib import Path

# Guard: manifest must exist and be non-empty before splitting
manifest_path = Path("/content/outputs/manifests/FFPP/manifest.csv")
if not manifest_path.exists():
    raise FileNotFoundError("manifest.csv not found — re-run Step 5 (extract frames) first")

manifest = pd.read_csv(manifest_path)
if len(manifest) == 0:
    raise RuntimeError("manifest.csv is empty — re-run Step 5 and fix the issue it reports")

print(f"Manifest has {len(manifest)} videos — proceeding to split")

!python /content/skripsi/deepfake_hybrid/scripts/build_splits.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --val-size 0.15 \
    --test-size 0.15 \
    --seed 42

# Sanity check splits
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f"/content/outputs/manifests/FFPP/{split}.csv")
    print(f"{split}: {len(df)} videos | real={df['label'].eq(0).sum()} fake={df['label'].eq(1).sum()}")

## Step 7 — Compute FFT cache
Only needed for `freq`, `hybrid`, and `early_fusion` models. Skip if training `spatial` only.

In [ ]:
!python /content/skripsi/deepfake_hybrid/scripts/compute_fft_cache.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --num-workers 2

## Step 8 — Train

Choose which model to train by running the corresponding cell.
All four cells can be run sequentially for the full experiment.

In [ ]:
# --- Spatial-only (XceptionNet baseline) ---
!python /content/skripsi/deepfake_hybrid/scripts/train.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --model spatial \
    --seed 42 \
    --pretrained

In [ ]:
# --- Frequency-only (FreqCNN baseline) ---
!python /content/skripsi/deepfake_hybrid/scripts/train.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --model freq \
    --seed 42

In [ ]:
# --- Hybrid two-branch (main model) ---
!python /content/skripsi/deepfake_hybrid/scripts/train.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --model hybrid \
    --seed 42 \
    --pretrained

In [ ]:
# --- Early fusion (alternative fusion) ---
!python /content/skripsi/deepfake_hybrid/scripts/train.py \
    --config "{CONFIG_PATH}" \
    --dataset FFPP \
    --model early_fusion \
    --seed 42 \
    --pretrained

## Step 9 — Evaluate checkpoints

In [ ]:
import json, glob

# Evaluate all trained checkpoints on the FFPP test set
for model_type in ['spatial', 'freq', 'hybrid', 'early_fusion']:
    ckpt = f"/content/outputs/runs/{model_type}_FFPP_seed42/best.pt"
    if not Path(ckpt).exists():
        print(f"[SKIP] No checkpoint for {model_type}")
        continue
    print(f"\n=== Evaluating {model_type} ===")
    !python /content/skripsi/deepfake_hybrid/scripts/eval.py \
        --config "{CONFIG_PATH}" \
        --dataset FFPP \
        --model "{model_type}" \
        --checkpoint "{ckpt}" \
        --seed 42 \
        --pretrained

In [ ]:
# Print a summary table of all results
import json
from pathlib import Path

results_dir = Path("/content/outputs/tables")
rows = []
for f in sorted(results_dir.glob("*.json")):
    data = json.loads(f.read_text())
    rows.append({"experiment": f.stem, **data})

if rows:
    import pandas as pd
    df = pd.DataFrame(rows).set_index("experiment")
    pd.set_option('display.float_format', '{:.4f}'.format)
    print(df.to_string())
else:
    print("No results yet.")

## Step 10 — Save everything back to Drive

**Important:** Colab resets when the session ends. Always run this cell before closing.

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size` to `8` in Step 4 |
| `EmptyDataError: No columns to parse` in Step 6 | manifest.csv is empty — re-run Step 5; check the debug cell output to see if `.mp4` files exist and if OpenCV can open them |
| `FileNotFoundError: manifest` in Step 6 | Re-run Step 5 (extract frames) first |
| Step 3 shows `MISSING` for real/fake paths | Your Drive shortcut points to the wrong folder — it must be the top-level folder containing both `original_sequences/` and `manipulated_sequences/` |
| Debug cell shows `OpenCV can open ...: False` | OpenCV can't decode the videos — make sure Step 2 installed `ffmpeg` (`apt-get install -qq ffmpeg libavcodec-extra`) and restart the runtime, then re-run from Step 2 |
| Debug cell shows "No .mp4 files found" | The copy in Step 3 failed silently — re-run Step 3 and wait for it to finish |
| `extract_frames.py` finds 0 videos | Check that `FFPP_IN_DRIVE` path is correct and `.mp4` files exist inside `original_sequences/youtube/c23/videos/` and `manipulated_sequences/*/c23/videos/` |
| All videos have `unknown label` warning | Folder names don't match keywords — verify the subfolders are exactly `original_sequences`, `Deepfakes`, and `Face2Face` |
| Real count is 0, only fakes found | `original_sequences/youtube/c23/videos/` folder is missing or empty |
| Session disconnects mid-training | Outputs in `/content/` are lost — next time copy from Drive and resume |
| Slow frame copy from Drive | Normal — Drive I/O is ~30 MB/s. Let Step 3 finish before proceeding |

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size` to `8` in Step 4 |
| `FileNotFoundError: manifest` | Re-run Steps 5 & 6 |
| Step 3 shows `MISSING` for real/fake paths | Your Drive shortcut points to the wrong folder — it must be the top-level folder containing both `original_sequences/` and `manipulated_sequences/` |
| `extract_frames.py` finds 0 videos | Check that `FFPP_IN_DRIVE` path is correct and that `.mp4` files exist inside `original_sequences/youtube/c23/videos/` and `manipulated_sequences/*/c23/videos/` |
| All videos have `unknown label` warning | Folder names don't match keywords — verify the subfolders are exactly `original_sequences`, `Deepfakes`, and `Face2Face` (case-sensitive) |
| Real count is 0, only fakes found | `original_sequences/youtube/c23/videos/` folder is missing or empty |
| Session disconnects mid-training | Outputs in `/content/` are lost — next time copy from Drive and resume |
| Slow frame copy from Drive | Normal — Drive I/O is ~30 MB/s. Let Step 3 finish before proceeding |